# Bridge Dashboard Generator

This notebook generates an interactive HTML dashboard for bridge engineers by aggregating data from multiple sources:

- **TIMs** - Transportation Information Management System
- **Assetwise** - Asset management data
- **BrR** - Bridge Reporting system
- **Historic Resources** - Historical bridge information

The dashboard provides comprehensive summaries and visualizations to support bridge inspection and maintenance decision-making.

In [1]:
# User Input Parameters
district = 6  # e.g., "1", "2", "3", etc.

# Gathering Data

Start with TIMs since it's the easiest to query and filter results from,

In [2]:
from src.civilpy.state.ohio.DOT.TIMS import get_bridge_sfns_by_district

In [3]:
# Define the endpoint and filter

district = 6  # Update the district here to change the bridges listed

# Call the function
d6_bridges = get_bridge_sfns_by_district(district=6)

Querying API for District 06...
URL: https://tims.dot.state.oh.us/ags/rest/services/Assets/Bridge_Inventory/MapServer/0/query
Filter: DISTRICT = '06'

Success! Found 4632 features.


In [4]:
# Sample 5 of the returned bridges to make sure we got SFNs
d6_bridges[-5:]

['8060014', '8060016', '8061662', '8062064', '8062641']

# Gather Assetwise Data

[Endpoint Documentation](https://ohiodot-it-api.bentley.com/swagger/index.html)

In [5]:
from civilpy.state.ohio.DOT.AssetWise import get_assetwise_secrets, base_url

In [6]:
username, password = get_assetwise_secrets()

In [7]:
from src.civilpy.state.ohio.DOT.AssetWise import get_bridge_by_sfn

In [8]:
# The first bridge in the list can be retrievied by asking for index "0"
d6_bridges[0]

'2100053'

In [9]:
single_bridge = get_bridge_by_sfn(d6_bridges[0])

In [10]:
from civilpy.state.ohio.DOT.AssetWise import get_elements_for_asset

In [11]:
bridge_elements = get_elements_for_asset(base_url, username, password, single_bridge['as_id'])

In [12]:
for element in bridge_elements:
    element_data = element
    
    output = f"""
    --- Element Details ---
    Element Name: {element_data.get('se_id', 'N/A')}: {element_data.get('se_name', 'N/A')}
    Structure ID: {element_data.get('assetId', 'N/A')}
    Sub-Asset ID (se_as_id): {element_data.get('se_as_id', 'N/A')}
    Environment ID: {element_data.get('environmentId', 'N/A')}
    Segment Asset ID: {element_data.get('segmentAssetId', 'N/A')}
    
    Quantities & States:
      - Total Quantity: {element_data.get('totalQuantity', 'N/A')}
      - Condition State 1: {element_data.get('state1', 'N/A')}
      - Condition State 2: {element_data.get('state2', 'N/A')}
      - Condition State 3: {element_data.get('state3', 'N/A')}
      - Condition State 4: {element_data.get('state4', 'N/A')}
      - Condition State 5: {element_data.get('state5', 'N/A')}
    """
    
    print(output)


    --- Element Details ---
    Element Name: 241: Reinforced Concrete Culvert
    Structure ID: 78577
    Sub-Asset ID (se_as_id): 108239
    Environment ID: 3
    Segment Asset ID: 108239
    
    Quantities & States:
      - Total Quantity: 111
      - Condition State 1: 111
      - Condition State 2: 0
      - Condition State 3: 0
      - Condition State 4: 0
      - Condition State 5: 0
    

    --- Element Details ---
    Element Name: 835: Culvert End Treatment
    Structure ID: 78577
    Sub-Asset ID (se_as_id): 108238
    Environment ID: 3
    Segment Asset ID: 108238
    
    Quantities & States:
      - Total Quantity: 6
      - Condition State 1: 4
      - Condition State 2: 2
      - Condition State 3: 0
      - Condition State 4: 0
      - Condition State 5: 0
    

    --- Element Details ---
    Element Name: 845: Roadway Over Structure
    Structure ID: 78577
    Sub-Asset ID (se_as_id): 108237
    Environment ID: 3
    Segment Asset ID: 108237
    
    Quantities & 

# Get D6 Bridges w/ Their Inspection Dates

In [13]:
import pandas as pd
from civilpy.state.ohio.DOT.AssetWise import get_inspection_reports_for_asset, get_bridge_by_sfn
from src.civilpy.state.ohio.DOT.AssetWise import get_all_approved_inspections

In [25]:
import json
import requests
from pathlib import Path
from datetime import datetime
from requests.auth import HTTPBasicAuth
from typing import List, Dict, Union

def get_inspection_schedules_for_asset(as_id: int, api_type: str = "api") -> List[Dict]:
    """
    Retrieves the inspection schedule(s) for a specific asset.
    Uses the GET /api/InspectionSchedule/GetSchedulesForAsset/{as_id} endpoint.

    Args:
        as_id (int): The unique ID of the asset.
        api_type (str): The API type, typically 'api'. Defaults to 'api'.

    Returns:
        List[Dict]: A list of dictionaries, each representing an AssetInspectionTypeSchedule.
                    Returns an empty list if no data is found or an error occurs.
    """
    # 1. Get auth secrets
    username, password = get_assetwise_secrets()
    
    # 2. Construct URL
    api_url = f"{base_url}/{api_type}/InspectionSchedule/GetSchedulesForAsset/{as_id}"
    
    # 3. Set headers and auth
    headers = {"Accept": "application/json"}
    auth = HTTPBasicAuth(username, password)
    
    # 4. Use a robust try/except block
    response = None 
    try:
        response = requests.get(api_url, headers=headers, auth=auth)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        data = response.json()
        
        if data.get("success"):
            # The 'data' field contains the list of schedule objects
            return data.get("data", [])
        else:
            print(f"API returned an error for asset {as_id}: {data.get('errorMessage', 'Unknown error')}")
            return []
            
    except requests.exceptions.RequestException as req_err:
        status_code = response.status_code if response is not None else "N/A"
        response_text = response.text if response is not None else "N/CSS"
        print(f"An HTTP request error occurred for asset {as_id}: {req_err} (Status Code: {status_code}, Response: {response_text})")
        return []
    except ValueError: # Handle JSON decode error
        print(f"Failed to decode JSON response for asset ID {as_id}.")
        return []

In [32]:
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from tqdm.auto import tqdm  # Use tqdm.auto for better Jupyter/console detection
from concurrent.futures import ThreadPoolExecutor, as_completed # <-- NEW: For threading

In [39]:
def process_bridge_sfn(bridge_sfn: str) -> tuple[dict, dict | None] | None:
    """
    Worker function to process a single bridge SFN.
    This function is designed to be run in a thread pool.
    On success, it returns a tuple: (row_data_for_df, full_report_json).
    On failure, it prints an error and returns None.
    """
    try:
        # 1. Get the full bridge details object from the SFN
        bridge_details = get_bridge_by_sfn(bridge_sfn)
        if not bridge_details:
            print(f"Skipping SFN {bridge_sfn}: Could not find bridge details.")
            return None

        # 2. Extract only the asset ID
        asset_id = bridge_details.get('as_id')
        if not asset_id:
            print(f"Skipping SFN {bridge_sfn}: No asset ID found in details.")
            return None
            
        # 3. Get ONLY the single most recent report.
        inspection_reports = get_inspection_reports_for_asset(asset_id, reports_to_return=1)

        # 4. Initialize variables
        next_due_date_str = None
        current_inspection_date_str = None
        all_inspection_types = set()
        inspector_id = None
        report_sections = []
        report_url = None
        full_report_data = None # This will hold the full report JSON
        
        if inspection_reports:
            # 5. Process the single report (inspection_reports[0])
            report = inspection_reports[0]
            full_report_data = report # Store the full JSON
            
            # --- Extract new fields ---
            inspector_id = report.get('inspectorID')
            report_url = report.get('reportURL')
            
            # Get all form names as the "sections"
            forms_list = report.get('forms', [])
            if forms_list:
                report_sections = [form.get('fm_name') for form in forms_list if form.get('fm_name')]
            # ---
            
            # Filter for "Approved" reports only
            if report.get('reportStatusText') == 'Approved':
                inspection_type_name = ''
                
                # Get inspection type name
                types_list = report.get('inspectionTypes', [])
                if types_list:
                    inspection_type_name = types_list[0].get('it_name', '')
                    if inspection_type_name:
                        all_inspection_types.add(inspection_type_name)
                
                # Get the inspection date string
                date_str = report.get('ast_inspection_date')
                if date_str:
                    try:
                        # Parse the ISO format date string
                        inspection_date = datetime.fromisoformat(date_str)
                        
                        # Set Current Inspection Date
                        current_inspection_date_str = inspection_date.strftime('%Y-%m-%d')
                        
                        # Calculate Next Inspection Due *only if* it was Routine
                        if 'Routine' in inspection_type_name:
                            next_due_date_dt = inspection_date + relativedelta(years=2)
                            next_due_date_str = next_due_date_dt.strftime('%Y-%m-%d')
                            
                    except (ValueError, TypeError):
                        # Skip if the date string is bad
                        print(f"Warning: Could not parse date '{date_str}' for asset {asset_id}")

        # 6. Prepare the row for the dataframe
        row_data = {
            'Bridge ID': bridge_sfn, 
            'Next Inspection Due': next_due_date_str,
            'Current Inspection Date': current_inspection_date_str,
            'Inspector ID': inspector_id,
            'Report URL': report_url,
            'Inspection Types': list(all_inspection_types),
            'Report Sections': report_sections
        }
        
        return (row_data, full_report_data) # Return the tuple

    except Exception as e:
        # Handle potential API errors or other issues
        print(f"Error processing bridge {bridge_sfn}: {e}")
        return None # Return None on failure

In [41]:
# These lists will hold the data
bridge_data_list = []
full_report_data_list = [] # <-- NEW: To store the full JSON
MAX_THREADS = 20 # As requested by user

print(f"Starting to process {len(d6_bridges)} bridges with {MAX_THREADS} threads...")

# Use ThreadPoolExecutor to run tasks in parallel
with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    # Submit all tasks to the executor
    # Create a dictionary mapping {future: bridge_sfn}
    futures = {executor.submit(process_bridge_sfn, bridge): bridge for bridge in d6_bridges}
    
    # Use as_completed to process results as they finish
    # Wrap in tqdm for a real-time progress bar
    for future in tqdm(as_completed(futures), total=len(d6_bridges), desc="Processing D6 Bridges"):
        result = future.result()
        
        # --- MODIFIED: Unpack the new tuple result ---
        if result is not None:
            row_data, report_json = result
            bridge_data_list.append(row_data)
            if report_json:
                full_report_data_list.append(report_json)
        # ---

print(f"\nSuccessfully processed {len(bridge_data_list)} bridges.")
# You now also have 'full_report_data_list' available to use

# --- Loop Finished ---

Starting to process 4632 bridges with 20 threads...


Processing D6 Bridges:   0%|          | 0/4632 [00:00<?, ?it/s]

Error processing bridge 2560404: 404 Client Error: Not Found for url: https://ohiodot-it-api.bentley.com/api/Asset/GetAssetByAsCode/2560404?IncludeCoordinates=True&IncludeParent=False

Successfully processed 4631 bridges.

--- DataFrame Created Successfully (top 5 rows) ---
     Bridge ID Next Inspection Due Current Inspection Date  Inspector ID  \
379    2100010                None              2025-03-26         454.0   
101    2100045          2027-02-11              2025-02-11         454.0   
7      2100053          2027-03-26              2025-03-26        1245.0   
70     2100118          2027-03-26              2025-03-26         454.0   
1627   2100127                None              2025-05-15        1245.0   

                                             Report URL  Inspection Types  \
379   https://ohiodot-it.bentley.com/inspection.aspx...    [Ohio Cursory]   
101   https://ohiodot-it.bentley.com/inspection.aspx...         [Routine]   
7     https://ohiodot-it.bentley.com/

In [ ]:
# Create the dataframe
df_inspections = pd.DataFrame(bridge_data_list)

# Sort by Bridge ID to maintain a consistent order
if not df_inspections.empty:
    df_inspections.sort_values(by='Bridge ID', inplace=True)

print("\n--- DataFrame Created Successfully (top 5 rows) ---")
print(df_inspections.head())


# --- Calculate and display inspections due in the next 24 months ---

print("\n--- Calculating Monthly Inspection Workload ---")

# 1. Define the 24-month window starting today (Nov 6, 2025)
today = datetime(2025, 11, 6)
start_date = today
end_date = today + relativedelta(months=24)

# 2. Ensure 'Next Inspection Due' is in datetime format for comparison
df_inspections['due_date_dt'] = pd.to_datetime(df_inspections['Next Inspection Due'], errors='coerce')

# 3. Filter the DataFrame to only include bridges due in our window
df_upcoming = df_inspections.dropna(subset=['due_date_dt'])
df_upcoming = df_upcoming[
    (df_upcoming['due_date_dt'] >= start_date) &
    (df_upcoming['due_date_dt'] < end_date)
]

if df_upcoming.empty:
    print("No inspections found due in the next 24 months.")
else:
    # 4. Create a 'Year-Month' column to group by
    df_upcoming_copy = df_upcoming.copy()
    df_upcoming_copy['Year-Month'] = df_upcoming_copy['due_date_dt'].dt.to_period('M')

    # 5. Group by the new 'Year-Month' column and get the size (count)
    monthly_counts = df_upcoming_copy.groupby('Year-Month').size()

    # 6. Create a full 24-month index to show months with 0 inspections
    all_months_index = pd.period_range(start=today, periods=24, freq='M')

    # 7. Reindex our counts to match the full 24-month index, filling gaps with 0
    monthly_summary = monthly_counts.reindex(all_months_index, fill_value=0)
    
    # 8. Set a clear name for the data series
    monthly_summary.name = "Inspections Due"

    print("\n--- Inspections Due in Next 24 Months ---")
    print(monthly_summary)

print("\n--- Script Finished ---")

In [43]:
import pandas as pd
from pathlib import Path
import ast # <-- Add this import for loading
import os

# --- Add these two functions to your script ---

def save_inspections_to_csv(df_to_save: pd.DataFrame, file_name: str = "d6_inspection_data.csv"):
    """
    Saves the DataFrame to a CSV file at the specified path.
    """
    # Define the full path using pathlib
    base_path = Path(r"C:\Users\dparks1\PycharmProjects\civilpy\res\inspections")
    full_filepath = base_path / file_name
    
    print(f"\nAttempting to save data to: {full_filepath}")
    
    try:
        # Create the directory if it doesn't exist
        os.makedirs(base_path, exist_ok=True)
        
        # Save to CSV, index=False is important to avoid an extra column
        df_to_save.to_csv(full_filepath, index=False)
        
        print(f"--- Successfully saved data to {full_filepath} ---")
        
    except PermissionError:
        print(f"Error: Permission denied. Could not write to {full_filepath}")
    except OSError as e:
        print(f"Error writing to file: {e}")

In [ ]:
# Sort by Bridge ID to maintain a consistent order
if not df_inspections.empty:
    df_inspections.sort_values(by='Bridge ID', inplace=True)

print("\n--- DataFrame Created Successfully (top 5 rows) ---")
print(df_inspections.head())

# --- NEW: SAVE THE DATAFRAME ---
save_inspections_to_csv(df_inspections)

In [ ]:
def load_inspections_from_csv(file_name: str = "d6_inspection_data.csv") -> pd.DataFrame | None:
    """
    Loads inspection data from a CSV file.
    
    This function specifically converts the 'Inspection Types' and 'Report Sections'
    columns from strings back into Python lists.
    """
    # Define the full path
    base_path = Path(r"C:\Users\dparks1\PycharmProjects\civilpy\res\inspections")
    full_filepath = base_path / file_name
    
    print(f"\nAttempting to load data from: {full_filepath}")
    
    try:
        # Define converters to handle the list columns
        converters = {
            'Inspection Types': ast.literal_eval,
            'Report Sections': ast.literal_eval
        }
        
        df_loaded = pd.read_csv(full_filepath, converters=converters)
        
        print(f"--- Successfully loaded data from {full_filepath} ---")
        return df_loaded
        
    except FileNotFoundError:
        print(f"Error: File not found at {full_filepath}")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: File is empty at {full_filepath}")
        return None
    except Exception as e:
        print(f"An error occurred during loading: {e}")
        return None

In [28]:
# 1. Get the full bridge details object from the SFN
bridge_details = get_bridge_by_sfn(bridge)
if not bridge_details:
    print(f"Skipping SFN {bridge}: Could not find bridge details.")

In [29]:
# 2. Extract only the asset ID
asset_id = bridge_details.get('as_id')
if not asset_id:
    print(f"Skipping SFN {bridge}: No asset ID found in details.")

In [31]:
bridge_details

{'as_id': 25698,
 'as_name': 'UNI-POST-0008 _(8062641)',
 'as_code': '8062641',
 'as_guid': 'd26c59ce-0fda-4dd3-b9c9-cd119b7f5cf9',
 'rt_id': 1,
 'as_parent_id': None,
 'as_parent': None,
 'as_asset_def': 0,
 'coordinates': {'usesDegMinSec': False,
  'latitude': '40.107331',
  'longitude': '-83.17185',
  'latitude_Degree': 40.0,
  'latitude_Minute': 6.0,
  'latitude_Seconds': 26.0,
  'longitude_Degree': -83.0,
  'longitude_Minute': -10.0,
  'longitude_Second': -18.0},
 'as_root': False,
 'as_deleted': False,
 'at_id': 1,
 'as_child_rt_id': 1,
 'as_child_at_id': None,
 'as_summary': False,
 'as_is_asset_view': False,
 'asset_status_id': 4,
 'as_corridor': False,
 'as_last_snapshot_date': None,
 'as_is_internal': False,
 'as_sub_asset_parent': False,
 'as_last_update_date': '2025-11-06T05:29:36.34',
 'as_default_file_set_date': '2025-10-06T14:07:07.413',
 'as_stage_pontis': False,
 'as_federal_submission_type': 0,
 'asset_segment_map': None,
 'corridor': None,
 'as_created_date': '2020-0

In [ ]:
# 3. --- NEW FAST LOGIC ---
# Get the inspection schedules for this asset
schedules = get_inspection_schedules_for_asset(asset_id)

next_due_date_str = None

if schedules:
    # 4. Find the "Routine" schedule
    # Note: We are assuming the name is 'Routine' or 'Routine (SNBI)'
    # We also assume 'ais_next_date' is the key for the next due date